In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Citirea datelor
data = pd.read_csv("../data/FinalDatasets/dataset_3stations_interpolated.csv", parse_dates=['start', 'end'])


In [49]:
data

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,latitude,location
0,2022-01-02 00:00:00,2022-01-02 01:00:00,46.48,42.360000,30.20,4.8,100.0,7.2,1021.3,26.036694,44.447275,Crangasi
1,2022-01-02 00:00:00,2022-01-02 01:00:00,43.33,32.427825,44.18,4.8,100.0,7.2,1021.3,26.127289,44.444925,Piata Obor
2,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,44.435044,Universitate
3,2022-01-02 01:00:00,2022-01-02 02:00:00,33.50,30.730000,18.84,3.6,100.0,3.6,1021.8,26.036694,44.447275,Crangasi
4,2022-01-02 01:00:00,2022-01-02 02:00:00,10.62,7.947923,31.10,3.6,100.0,3.6,1021.8,26.127289,44.444925,Piata Obor
...,...,...,...,...,...,...,...,...,...,...,...,...
19852,2022-12-21 04:00:00,2022-12-21 05:00:00,24.68,18.597614,18.29,-3.3,99.0,10.8,1027.5,26.127289,44.444925,Piata Obor
19853,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,44.435044,Universitate
19854,2022-12-21 05:00:00,2022-12-21 06:00:00,21.42,17.410000,16.81,-3.1,98.0,7.2,1027.1,26.036694,44.447275,Crangasi
19855,2022-12-21 05:00:00,2022-12-21 06:00:00,23.14,17.437147,21.16,-3.1,98.0,7.2,1027.1,26.127289,44.444925,Piata Obor


In [50]:
# import pandas as pd
# import numpy as np
# from scipy.spatial import distance

# # Încărcăm datele
# data = pd.read_csv("../data/FinalDatasets/dataset_3stations_interpolated.csv", parse_dates=["start", "end"])

# # Extragem ziua și ora
# data["day_of_week"] = data["start"].dt.dayofweek
# data["hour"] = data["start"].dt.hour
# data["month"] = data["start"].dt.month

# # Alegem ce vrem să interpolăm
# pollution_features = ["pm10", "pm2_5", "no2"]

# # Funcția de IDW
# def idw_interpolation(lat, lon, hour, day_of_week, month, data, k=5, power=2):
#     filtered = data[
#         (data["hour"] == hour) &
#         (data["day_of_week"] == day_of_week) &
#         (data["month"] == month)
    # ].copy()

#     # 🧮 Calculează distanța vectorial
#     coords = filtered[["latitude", "longitude"]].values
#     target = np.array([lat, lon])
#     filtered["distance"] = np.linalg.norm(coords - target, axis=1)

#     # 🛑 Evită punctele la distanță 0
#     filtered = filtered[filtered["distance"] > 0]

#     # 🔍 Cei mai apropiați k vecini
#     nearest = filtered.nsmallest(k, "distance")

#     # 📈 Interpolare pe fiecare poluant
#     predictions = {}
#     for pol in pollution_features:
#         weights = 1 / (nearest["distance"] ** power)
#         interpolated = np.sum(weights * nearest[pol]) / np.sum(weights)
#         predictions[pol] = interpolated

#     return predictions


# # Exemplu de interpolare: punct nou în luna decembrie, miercuri, ora 8
# test_lat = 44.4350433	
# test_lon = 26.0982971	
# test_hour = 23
# test_day = 5  # miercuri
# test_month = 5

# result = idw_interpolation(test_lat, test_lon, test_hour, test_day, test_month, data)
# print(result)



IndentationError: unexpected indent (3133945658.py, line 22)

In [89]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Încarcă datele
data = pd.read_csv("../data/FinalDatasets/dataset_3stations_interpolated.csv", parse_dates=["start", "end"])

# Feature engineering
data["hour"] = data["start"].dt.hour
data["day"] = data["start"].dt.day
data["month"] = data["start"].dt.month
data["year"] = data["start"].dt.year
data["day_of_week"] = data["start"].dt.dayofweek

# Definim features și targets
features = ["latitude", "longitude", "hour", "day", "month", "year", "day_of_week", "temperature", "humidity", "wind_speed", "pressure"]
targets = ["pm10", "pm2_5", "no2"]

X = data[features]
y = data[targets]

# Împărțim în train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Multi-output model
base_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model = MultiOutputRegressor(base_model)
model.fit(X_train, y_train)

# Evaluare
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

# Rezultate
for i, pol in enumerate(targets):
    print(f"\n📊 {pol.upper()} - Metrics")
    print(f"MAE:  {mae[i]:.2f}")
    print(f"RMSE: {rmse[i]:.2f}")
    print(f"R²:   {r2[i]:.3f}")



📊 PM10 - Metrics
MAE:  6.15
RMSE: 9.27
R²:   0.616

📊 PM2_5 - Metrics
MAE:  3.78
RMSE: 5.62
R²:   0.665

📊 NO2 - Metrics
MAE:  8.16
RMSE: 11.13
R²:   0.693


In [90]:
# Exemplu: predicție pentru o locație nouă
sample_input = pd.DataFrame([{
    "latitude": 44.43523472296773,     # locație din București
    "longitude": 26.097022283439195,
    "hour": 18,
    "day": 25,
    "month": 5,
    "year":2025,
    "day_of_week": 6,
    "temperature": 14,
    "humidity": 85.0,
    "wind_speed": 2.0,
    "pressure": 1001.0
}])
prediction = model.predict(sample_input)
print("Predicții pentru PM10, PM2.5, NO2:", prediction[0])


Predicții pentru PM10, PM2.5, NO2: [17.645138 10.046217 31.688444]


In [91]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Încarcă datele
data = pd.read_csv("../data/FinalDatasets/dataset_3stations_interpolated.csv", parse_dates=["start", "end"])

# 2. Feature Engineering
data["hour"] = data["start"].dt.hour
data["day"] = data["start"].dt.day
data["month"] = data["start"].dt.month
data["year"] = data["start"].dt.year
data["day_of_week"] = data["start"].dt.dayofweek
# data["is_weekend"] = data["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)

# 3. Features & Targets
features = [
    "latitude", "longitude", "hour", "day", "month", "year",
    "day_of_week", "temperature", "humidity", "wind_speed", "pressure"
]
targets = ["pm10", "pm2_5", "no2"]

X = data[features]
y = data[targets]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Hyperparameter Tuning
param_grid = {
    'estimator__n_estimators': [100, 200],
    'estimator__max_depth': [3, 5, 7],
    'estimator__learning_rate': [0.01, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

base_model = XGBRegressor(random_state=42)
multi_model = MultiOutputRegressor(base_model)

grid = GridSearchCV(
    estimator=multi_model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=2,
    n_jobs=-1
)

# 🔁 Poți reduce la X_train[:5000] dacă durează prea mult
grid.fit(X_train, y_train)

print("📌 Best Parameters:", grid.best_params_)
print("📉 Best MAE (neg):", -grid.best_score_)


Fitting 3 folds for each of 48 candidates, totalling 144 fits
📌 Best Parameters: {'estimator__colsample_bytree': 1.0, 'estimator__learning_rate': 0.1, 'estimator__max_depth': 7, 'estimator__n_estimators': 200, 'estimator__subsample': 0.8}
📉 Best MAE (neg): 4.506323019663493


In [92]:


from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Creează modelul cu cei mai buni parametri
best_model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=200,
        max_depth=7,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        random_state=42
    )
)

best_model.fit(X_train, y_train)

# 3️⃣ Preziceri
y_pred = best_model.predict(X_test)

# 4️⃣ Metrice de evaluare
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

# 5️⃣ Afișare rezultate
for i, pol in enumerate(["pm10", "pm2_5", "no2"]):
    print(f"\n📊 {pol.upper()} - Metrics")
    print(f"MAE:  {mae[i]:.2f}")
    print(f"RMSE: {rmse[i]:.2f}")
    print(f"R²:   {r2[i]:.3f}")



📊 PM10 - Metrics
MAE:  4.18
RMSE: 6.92
R²:   0.786

📊 PM2_5 - Metrics
MAE:  2.64
RMSE: 4.04
R²:   0.827

📊 NO2 - Metrics
MAE:  6.13
RMSE: 8.51
R²:   0.820


In [109]:
def predict_pollution(model, lat, lon, year, month, day, hour, temperature, humidity, wind_speed, pressure):
    dt = datetime(year, month, day)
    day_of_week = dt.weekday()

    input_features = np.array([[
        lat, lon, hour, day, month, year,
        day_of_week, temperature, humidity, wind_speed, pressure
    ]])

    prediction = model.predict(input_features)[0]

    return {
        "pm10": round(prediction[0], 2),
        "pm2_5": round(prediction[1], 2),
        "no2": round(prediction[2], 2)
    }


In [115]:
from datetime import datetime
import numpy as np

result = predict_pollution(
    model=best_model,
    lat=44.447275,
    lon=26.036694,
    year=2025,
    month=5,
    day=23,
    hour=8,
    temperature=26.4,
    humidity=44.0,
    wind_speed=7.9,
    pressure=1007.6
)

print("Pred:", result)


Pred: {'pm10': 28.11, 'pm2_5': 12.75, 'no2': 21.78}


In [120]:
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

ordered_model = XGBRegressor(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=1.0,
    random_state=42
)

# ordinea în care sunt legate targeturile
# Exemplu: 0 = pm10, 1 = pm2_5, 2 = no2
# Dacă vrei no2 → pm2_5 → pm10: ordinea = [2, 1, 0]
chain_model = RegressorChain(ordered_model, order=[2, 1, 0])

chain_model.fit(X_train, y_train)

y_pred = chain_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

for i, pol in enumerate(["pm10", "pm2_5", "no2"]):
    print(f"\n🔗 {pol.upper()} - Chain Metrics")
    print(f"MAE:  {mae[i]:.2f}")
    print(f"RMSE: {rmse[i]:.2f}")
    print(f"R²:   {r2[i]:.3f}")



🔗 PM10 - Chain Metrics
MAE:  4.72
RMSE: 7.73
R²:   0.732

🔗 PM2_5 - Chain Metrics
MAE:  2.88
RMSE: 4.40
R²:   0.795

🔗 NO2 - Chain Metrics
MAE:  6.13
RMSE: 8.51
R²:   0.820
